# Competitive Analysis

This worked example turns the 227 product detail files in `product-results/` into one table, removes the items that are not makeup foundations or are not priced in US dollars, and then looks at price, ratings, brands, positioning keywords, badges, and demand signals from the point of view of a MAC Cosmetics analyst.

:::{note} One snapshot

All files come from a single scrape of Amazon search results for "foundation" (sorted by Best Selling). The analysis describes that one moment. It cannot show trends, growth, or what caused a product to sell.

:::


---

## 📦 Step 1: Import Libraries


In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio

PLOTLY_TEMPLATE = "simple_white"
pio.templates.default = PLOTLY_TEMPLATE

In [2]:
pd.set_option("display.max_columns", 50)

---

## 📂 Step 2: Load & Parse JSON Files

Each file in `product-results/` is a detailed product page for one ASIN.
We navigate the nested structure, for example `data["product"]["price"]["currentPrice"]`.

Three fields matter for cleaning later:

- `price.symbol`: the currency of the price. Most are `$`, but a few listings were returned in other currencies.
- `breadcrumbs`: Amazon's category path, such as `Beauty & Personal Care > Makeup > Face > Foundation`. We keep the last element as `category`.
- `specification`: a list of `{"key": ..., "value": ...}` pairs. The `Brand Name` entry holds the brand.


In [3]:
PRODUCT_DIR = Path("product-results")


def spec_value(p: dict, key: str) -> str | None:
    """Return the value of one entry in the product's specification list."""
    for item in p.get("specification") or []:
        if item.get("key") == key:
            return item.get("value")
    return None


def pct(value: str | None) -> float | None:
    """Convert a string such as '78%' or '-5%' to a fraction."""
    if not value:
        return None
    return float(value.replace("%", "")) / 100


def parse_product(filepath: Path) -> dict | None:
    """Load one product JSON and return a flat dict of key fields."""
    with open(filepath, "r", encoding="utf-8") as f:
        data = json.load(f)

    p = data.get("product", {})
    if not p:
        return None

    # Price and currency
    price_obj = p.get("price") or {}

    # Reviews
    rev_obj = p.get("reviewsInfo") or {}
    stars = rev_obj.get("starRates") or {}

    # Badges
    badges = p.get("badges") or {}

    # Amazon category path, e.g. Beauty & Personal Care > Makeup > Face > Foundation
    breadcrumbs = p.get("breadcrumbs") or []

    # "Bought in past month" is a rounded label such as "400+" or "2K+"
    bought_raw = p.get("boughtInPastMonth") or ""
    bought_num = None
    if bought_raw:
        try:
            bought_num = int(bought_raw.replace("+", "").replace(",", "").replace("K", "000"))
        except ValueError:
            pass

    return {
        "asin": p.get("asin"),
        "title": p.get("title", ""),
        "amazon_url": p.get("url"),
        "brand": spec_value(p, "Brand Name"),
        "category_path": " > ".join(breadcrumbs),
        "category": breadcrumbs[-1] if breadcrumbs else None,
        "currency": price_obj.get("symbol"),
        "price": price_obj.get("currentPrice"),
        "before_price": price_obj.get("beforePrice"),
        "discount": pct(price_obj.get("discount")),
        "rating": rev_obj.get("rating"),
        "total_reviews": rev_obj.get("totalReviews"),
        "five_star_pct": pct(stars.get("fiveStars")),
        "four_star_pct": pct(stars.get("fourStars")),
        "three_star_pct": pct(stars.get("threeStars")),
        "two_star_pct": pct(stars.get("twoStars")),
        "one_star_pct": pct(stars.get("oneStars")),
        "seller": p.get("seller"),
        "is_best_seller": badges.get("bestSeller", False),
        "is_amazon_choice": badges.get("amazonChoice", False),
        "is_amazon_exclusive": badges.get("amazonExclusive", False),
        "is_amazon_prime": badges.get("amazonPrime", False),
        "is_limited_time_deal": badges.get("limitedTimeDeal", False),
        "bought_past_month": bought_num,
        "bullets": " ".join(p.get("featureBullets") or []),
        "description": p.get("description") or "",
        "is_available": p.get("isAvailable", False),
    }

In [4]:
records = []

# Sort the file list so the row order (and the CSV written below) is the same on every machine
for fp in sorted(PRODUCT_DIR.glob("*.json")):
    row = parse_product(fp)
    if row:
        records.append(row)

df = pd.DataFrame(records)
print(f"Loaded {len(df)} products from {PRODUCT_DIR}/")
print(f"Products with a 'Brand Name' in the specification: {df['brand'].notna().sum()}")

Loaded 227 products from product-results/
Products with a 'Brand Name' in the specification: 203


Some listings have no `specification` block, so they have no `Brand Name`. Many of them still start their title with a brand that appears elsewhere in the data (for example, "Maybelline Fit Me ..."). As a fallback, we match the start of the title against the brand names we already know, ignoring case. Anything left over is labelled `"Unknown"`.

:::{caution} Brand names are entered by sellers

The `Brand Name` field is not standardized. The same company can appear under slightly different names (for example, `Hush Makeup` and `Hush Makeup Foundation`). Treat counts for small brands as approximate.

:::


In [5]:
known_brands = sorted(df["brand"].dropna().unique(), key=len, reverse=True)


def brand_from_title(title: str) -> str:
    """Return the longest known brand that the title starts with, ignoring case."""
    for b in known_brands:
        if title.lower().startswith(b.lower()):
            return b
    return "Unknown"


missing = df["brand"].isna()
df.loc[missing, "brand"] = df.loc[missing, "title"].map(brand_from_title)

df.loc[missing, ["title", "brand"]].assign(title=lambda d: d["title"].str[:60])

,title,brand
0,"NIV, Foundation Study Bible, Leathersoft, Brow...",Unknown
1,SCP Foundation Artbook | Paperback Edition | B...,Unknown
7,Dermablend Cover Crème Full Coverage Foundatio...,Unknown
16,Foundation,Unknown
17,Second Foundation,Unknown
20,Maybelline Fit Me Dewy + Smooth Liquid Foundat...,MAYBELLINE
22,Foundation's Edge,Unknown
23,Forward the Foundation,Unknown
24,Foundation and Earth,Unknown
30,bareMinerals Original Loose Powder Foundation ...,bareMinerals


In [6]:
df[["asin", "brand", "category", "currency", "price", "rating", "total_reviews"]].head()

,asin,brand,category,currency,price,rating,total_reviews
0,0310441870,Unknown,Personal Growth,$,16.47,4.8,2981.0
1,1638380112,Unknown,NaN,$,24.45,4.8,76.0
2,B000IJLQOW,REVLON,Foundation,$,12.48,4.5,17834.0
3,B000QW6SXK,COVERGIRL,Foundation,$,8.98,4.5,12728.0
4,B000R8YZQA,Glo Skin Beauty,Powder,$,52.00,4.5,2767.0


Save the full parsed table (all 227 files, before any filtering) so it can be opened in a spreadsheet or reused without parsing the JSON again. The file is regenerated each time this notebook runs.

In [7]:
df.to_csv("foundation_products.csv", index=False)

---

## 🧹 Step 3: Data Cleaning & Feature Engineering


First, look at the most expensive items. Extreme values usually point to a data problem, so check the currency and category columns before deciding what to drop.

In [8]:
df.sort_values("price", ascending=False)[
    ["title", "currency", "price", "category"]
].head(10).assign(title=lambda d: d["title"].str[:50])

,title,currency,price,category
5,"COVERGIRL Smoothers Lightweight BB Cream, Fair...",PYG,45152.00,BB Creams
124,NYX PROFESSIONAL MAKEUP Bare With Me Blur Skin...,ARS,19575.50,Foundation
10,Mineral Fusion Pressed Powder Foundation - Mat...,MXN,593.85,Foundation
82,"Amazon Basics Smart Box Spring Bed Base, 5-Inc...",$,86.00,Box Springs
185,"No Makeup Makeup Foundation + Brush Duo, Cream...",$,80.00,Foundation
144,"EMODA 5 Inch Box Spring Queen Size Bed Base, H...",$,69.99,Box Springs
214,Hush Glow™ Foundation Cushion SPF 30 – Skincar...,CAD,59.93,Foundation
93,Estée Lauder Futurist Hydra Rescue Moisturizin...,$,55.00,Foundation
164,"No Makeup Makeup Foundation, Weightless Cream-...",$,55.00,Foundation
95,Lancôme Renergie Lift Makeup Foundation - Ligh...,$,53.55,Foundation


The two prices above 10,000 are not errors. One is in Paraguayan guaraní (`PYG`) and one in Argentine pesos (`ARS`). Other listings are in Mexican pesos, Canadian dollars, or pounds. Comparing them with US dollar prices would be meaningless.

The list also shows a box spring ("mattress foundation"). A search for "foundation" returns more than makeup. The two tables below count the currencies and the Amazon categories in the data.

In [9]:
df["currency"].value_counts(dropna=False)

currency
$      209
NaN     11
CAD      3
PYG      1
MXN      1
ARS      1
GBP      1
Name: count, dtype: int64

In [10]:
df["category_path"].value_counts(dropna=False)

category_path
Beauty & Personal Care > Makeup > Face > Foundation                                                                      159
Beauty & Personal Care > Makeup > Face > CC Creams                                                                        11
Beauty & Personal Care > Makeup > Face > BB Creams                                                                         9
Beauty & Personal Care > Makeup > Face > Powder                                                                            8
Beauty & Personal Care > Makeup > Face > Tinted Moisturizers                                                               6
Beauty & Personal Care > Makeup > Face > Foundation Primers                                                                6
Beauty & Personal Care > Makeup > Face > Concealers & Neutralizers                                                         5
Books > Science Fiction & Fantasy > Science Fiction > Hard Science Fiction                                     

We apply one rule before any analysis:

:::{important} Filtering rule

Keep a product only if

1. its price is in US dollars (`currency == "$"`), and
2. Amazon files it under `Beauty & Personal Care > Makeup > Face` in one of these categories: **Foundation, BB Creams, CC Creams, Tinted Moisturizers, or Powder**.

:::

The categories in rule 2 are products that are sold as a foundation or as a lighter foundation alternative. The rule removes books (Asimov's *Foundation*, a study Bible), box springs, hair care, nail coats, primers, concealers, palettes, and makeup removers. Items with no price are removed by rule 1.


In [11]:
FOUNDATION_CATEGORIES = ["Foundation", "BB Creams", "CC Creams", "Tinted Moisturizers", "Powder"]

df["is_usd"] = df["currency"].eq("$")
df["is_makeup_product"] = df["category_path"].str.startswith(
    "Beauty & Personal Care > Makeup > Face"
) & df["category"].isin(FOUNDATION_CATEGORIES)

print(f"All products:            {len(df)}")
print(f"Priced in US dollars:    {df['is_usd'].sum()}")
print(f"Makeup foundation items: {df['is_makeup_product'].sum()}")

df = df[df["is_usd"] & df["is_makeup_product"]].copy()
print(f"Kept for analysis:       {len(df)}")

All products:            227
Priced in US dollars:    209
Makeup foundation items: 193
Kept for analysis:       181


In [12]:
# Price tier buckets (left-closed, so "Under $10" means less than $10)
bins = [0, 10, 20, 30, 50, np.inf]
labels = ["Under $10", "$10–$20", "$20–$30", "$30–$50", "$50+"]
df["price_tier"] = pd.cut(df["price"], bins=bins, labels=labels, right=False)

# Keyword flags for positioning analysis
keyword_map = {
    "full_coverage": r"full.?coverage",
    "medium_coverage": r"medium.?coverage",
    "lightweight": r"light.?weight",
    "hydrating": r"hydrat",
    "matte": r"matte",
    "dewy": r"\bdew",
    "natural_finish": r"natural.?finish",
    "spf": r"\bspf",
    "long_wear": r"long.?wear|24.?hour|24hr",
    "skin_tint": r"skin.?tint",
    "foundation_stick": r"\bstick\b",
    "buildable": r"buildable",
}

combined_text = (df["title"] + " " + df["bullets"] + " " + df["description"]).str.lower()

for kw, pattern in keyword_map.items():
    df[kw] = combined_text.str.contains(pattern, regex=True).astype(int)

print("Cleaning done. Shape:", df.shape)
print("\nMissing values:")
print(df[["price", "rating", "total_reviews", "bought_past_month"]].isna().sum())

Cleaning done. Shape: (181, 42)

Missing values:
price                0
rating               2
total_reviews        2
bought_past_month    9
dtype: int64


---

## 📊 Step 4: Exploratory Analysis

### 4.1 Dataset Summary


In [13]:
df[["price", "rating", "total_reviews", "bought_past_month"]].describe().round(2)

,price,rating,total_reviews,bought_past_month
count,181.00,179.00,179.00,172.00
mean,21.41,4.28,8573.89,1498.84
std,14.27,0.36,18929.04,2706.31
min,4.92,2.00,3.00,200.00
25%,9.99,4.10,573.00,400.00
50%,15.98,4.40,2347.00,600.00
75%,31.20,4.50,8612.50,1000.00
max,80.00,4.80,154234.00,20000.00


### 4.2 Price Distribution: What Price Range Dominates?


In [14]:
fig = px.histogram(
    df,
    x="price",
    nbins=20,
    title="💰 Price Distribution of Best-Selling Foundation Products",
    labels={"price": "Price (USD)"},
    template=PLOTLY_TEMPLATE,
)
fig.update_layout(yaxis_title="# Products")
fig.show()

### 4.3 Price Tier Share


In [15]:
tier_counts = df["price_tier"].value_counts(sort=False).reset_index()
tier_counts.columns = ["price_tier", "count"]
tier_counts["share"] = (tier_counts["count"] / tier_counts["count"].sum()).round(3)
print(tier_counts.to_string(index=False))

fig = px.pie(
    tier_counts,
    names="price_tier",
    values="count",
    title="Price Tier Distribution of Best-Selling Foundations",
    color_discrete_sequence=px.colors.sequential.Oranges,
    template=PLOTLY_TEMPLATE,
    hole=0.35,
)
fig.update_traces(textposition="outside", textinfo="percent+label", sort=False)
fig.show()

price_tier  count  share
 Under $10     50  0.276
   $10–$20     66  0.365
   $20–$30     16  0.088
   $30–$50     42  0.232
      $50+      7  0.039


### 4.4 Rating Distribution


In [16]:
fig = px.histogram(
    df.dropna(subset=["rating"]),
    x="rating",
    nbins=20,
    title="Rating Distribution Across Best-Selling Foundations",
    labels={"rating": "Star Rating"},
    color_discrete_sequence=["green"],
    template=PLOTLY_TEMPLATE,
)
fig.update_layout(yaxis_title="# Products")
fig.show()

### 4.5 Do Higher-Rated Products Also Have More Reviews?


In [17]:
plot_df = df.dropna(subset=["rating", "total_reviews"]).copy()

fig = px.scatter(
    plot_df,
    x="rating",
    y="total_reviews",
    size="price",
    hover_name="title",
    hover_data={"brand": True, "price": True, "rating": True, "total_reviews": True},
    title="Rating vs. Total Reviews (bubble size = price)",
    labels={
        "rating": "Avg Star Rating",
        "total_reviews": "Total Reviews (log scale)",
        "price": "Price ($)",
    },
    template=PLOTLY_TEMPLATE,
    log_y=True,
)
fig.show()

# Review counts are highly skewed, so correlate rating with log10(reviews),
# and also report the rank-based Spearman correlation
pearson_log = plot_df["rating"].corr(np.log10(plot_df["total_reviews"]))
spearman = plot_df["rating"].corr(plot_df["total_reviews"], method="spearman")
print(f"Pearson correlation (rating vs. log10 reviews): {pearson_log:.3f}")
print(f"Spearman rank correlation (rating vs. reviews): {spearman:.3f}")

Pearson correlation (rating vs. log10 reviews): 0.521
Spearman rank correlation (rating vs. reviews): 0.412


### 4.6 Brand Dominance (Who Owns the Category?)


A "SKU" here is one Amazon listing (one ASIN). Some brands list several shades or sizes as separate ASINs, so SKU counts measure shelf presence in these search results, not sales.

In [18]:
brand_all = (
    df.groupby("brand", as_index=False)
    .agg(
        product_count=("asin", "count"),
        avg_price=("price", "mean"),
        avg_rating=("rating", "mean"),
        total_reviews=("total_reviews", "sum"),
    )
    .sort_values(["product_count", "total_reviews"], ascending=False)
)
print(f"Distinct brands: {len(brand_all)}")
print(f"Brands with a single listing: {(brand_all['product_count'] == 1).sum()}")

# Keep named brands with at least 3 listings so the chart is not decided by ties at 1 or 2
brand_df = brand_all[(brand_all["product_count"] >= 3) & (brand_all["brand"] != "Unknown")]

fig = px.bar(
    brand_df,
    x="product_count",
    y="brand",
    orientation="h",
    color="avg_rating",
    color_continuous_scale="RdYlGn",
    title="🏷️ Brands with 3+ Best-Selling Listings (color = avg rating)",
    labels={"product_count": "# of SKUs", "brand": "Brand", "avg_rating": "Avg Rating"},
    template=PLOTLY_TEMPLATE,
    text="product_count",
    height=600,
)
fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.show()

Distinct brands: 92
Brands with a single listing: 59


### 4.7 Price vs. Avg Rating by Brand


In [19]:
fig = px.scatter(
    brand_df,
    x="avg_price",
    y="avg_rating",
    size="total_reviews",
    text="brand",
    title="💵 Avg Price vs. Avg Rating by Brand (brands with 3+ listings; bubble size = total reviews)",
    labels={
        "avg_price": "Avg Price ($)",
        "avg_rating": "Avg Rating",
        "total_reviews": "Total Reviews",
    },
    template=PLOTLY_TEMPLATE,
    color="brand",
)
fig.update_traces(textposition="top center")
fig.update_layout(xaxis_tickprefix="$", showlegend=False)
fig.show()

### 4.8 Product Positioning: Keyword Frequency

> Which product attributes are most commonly called out in titles, bullets, and descriptions?

Each flag is 1 if the pattern appears anywhere in the product's title, feature bullets, or description. A flag records what the listing *says*, not what the product is.

In [20]:
kw_cols = list(keyword_map.keys())
kw_counts = df[kw_cols].sum().reset_index()
kw_counts.columns = ["keyword", "count"]
kw_counts = kw_counts.sort_values("count", ascending=True)

# Friendlier labels
label_map = {
    "full_coverage": "Full Coverage",
    "medium_coverage": "Medium Coverage",
    "lightweight": "Lightweight",
    "hydrating": "Hydrating",
    "matte": "Matte Finish",
    "dewy": "Dewy Finish",
    "natural_finish": "Natural Finish",
    "spf": "SPF / Sun Protection",
    "long_wear": "Long Wear / 24H",
    "skin_tint": "Skin Tint",
    "foundation_stick": "Stick Format",
    "buildable": "Buildable Coverage",
}
kw_counts["label"] = kw_counts["keyword"].map(label_map)

fig = px.bar(
    kw_counts,
    x="count",
    y="label",
    orientation="h",
    title=f"🔑 Product Positioning Keywords (out of {len(df)} products)",
    labels={"count": "# of Products Mentioning Keyword", "label": ""},
    color="count",
    color_continuous_scale="Reds",
    template=PLOTLY_TEMPLATE,
    text="count",
    height=600,
)
fig.update_layout(coloraxis_showscale=False)
fig.show()

### 4.9 Positioning by Price Tier


In [21]:
kw_cols_top = [
    "full_coverage",
    "lightweight",
    "hydrating",
    "matte",
    "natural_finish",
    "long_wear",
    "spf",
]

tier_kw = (
    df.dropna(subset=["price_tier"])
    .groupby("price_tier", observed=True)[kw_cols_top]
    .mean()
    .round(2)
    .reset_index()
)

tier_kw_melted = tier_kw.melt(
    id_vars="price_tier", var_name="keyword", value_name="share"
)
tier_kw_melted["keyword"] = tier_kw_melted["keyword"].map(
    {
        "full_coverage": "Full Coverage",
        "lightweight": "Lightweight",
        "hydrating": "Hydrating",
        "matte": "Matte",
        "natural_finish": "Natural Finish",
        "long_wear": "Long Wear / 24H",
        "spf": "SPF",
    }
)

fig = px.bar(
    tier_kw_melted,
    x="price_tier",
    y="share",
    color="keyword",
    barmode="group",
    title="🎯 Product Positioning Attributes by Price Tier",
    labels={
        "share": "Share of Products",
        "price_tier": "Price Tier",
        "keyword": "Attribute",
    },
    template=PLOTLY_TEMPLATE,
    color_discrete_sequence=px.colors.qualitative.Set2,
)
fig.update_layout(yaxis_tickformat=".0%")
fig.show()

### 4.10 Review Volume vs. Price: Social Proof at Different Price Points


In [22]:
box_df = df.dropna(subset=["price_tier", "total_reviews"]).copy()

fig = px.box(
    box_df,
    x="price_tier",
    y="total_reviews",
    color="price_tier",
    points="all",
    log_y=True,
    title="📣 Review Volume (Social Proof) by Price Tier",
    labels={"price_tier": "Price Tier", "total_reviews": "Total Reviews (log)"},
    template=PLOTLY_TEMPLATE,
    color_discrete_sequence=px.colors.sequential.Reds_r,
)
fig.update_layout(showlegend=False)
fig.show()

### 4.11 Attributes Often Linked to Younger Shoppers

The case scenario says younger shoppers favour lightweight, skin-tint, natural-finish, hydrating, and dewy formulas. We count how many of these five attributes each listing mentions and call it an "appeal score" (0 to 5).

:::{caution} This score is a hypothesis, not a measurement

The data has no information about who buys each product. The score shows how a listing is *marketed*. Whether these attributes actually attract Gen Z or Millennial shoppers would need survey, panel, or campaign data.

:::

In [23]:
genz_signals = ["lightweight", "skin_tint", "natural_finish", "hydrating", "dewy"]
df["genz_score"] = df[genz_signals].sum(axis=1)

# How common are high scores in each price tier?
score_by_tier = (
    df.groupby("price_tier", observed=True)
    .agg(products=("asin", "count"), score_3_plus=("genz_score", lambda s: (s >= 3).sum()))
    .assign(share_3_plus=lambda d: (d["score_3_plus"] / d["products"]).round(2))
)
print("Products scoring 3+ by price tier:")
print(score_by_tier.to_string())

genz_df = (
    df[df["genz_score"] > 0]
    .sort_values(["genz_score", "total_reviews"], ascending=False)
    .head(20)[["title", "brand", "price", "rating", "total_reviews", "genz_score"]]
)

fig = px.scatter(
    genz_df.dropna(subset=["total_reviews"]),
    x="price",
    y="rating",
    size="total_reviews",
    color="genz_score",
    hover_name="title",
    color_continuous_scale="Viridis",
    title="🧬 Appeal Score vs. Price & Rating (top 20 products by score)",
    labels={
        "price": "Price ($)",
        "rating": "Avg Rating",
        "genz_score": "Appeal Score",
        "total_reviews": "Total Reviews",
    },
    template=PLOTLY_TEMPLATE,
)
fig.update_layout(xaxis_tickprefix="$")
fig.show()

print("\nTop 10 products by appeal score:")
print(genz_df.head(10).assign(title=lambda d: d["title"].str[:50]).to_string(index=False))

Products scoring 3+ by price tier:
            products  score_3_plus  share_3_plus
price_tier                                      
Under $10         50            10          0.20
$10–$20           66            18          0.27
$20–$30           16             5          0.31
$30–$50           42            10          0.24
$50+               7             0          0.00



Top 10 products by appeal score:
                                             title             brand  price  rating  total_reviews  genz_score
IT Cosmetics CC+ Cream - Natural Full Coverage Fou      IT Cosmetics  31.20     4.5        32039.0           4
COVERGIRL Clean Fresh Nourishing Skin Tint, 550 Li         COVERGIRL  10.28     4.1        10061.0           4
JUNG SAEM MOOL Essential Skin Nuder Cushion | Natu    JUNG SAEM MOOL  22.40     4.1         3264.0           4
IT Cosmetics CC+ Nude Glow - Medium Coverage Found      IT Cosmetics  31.20     4.5         3009.0           4
Charlotte Tilbury Unreal Skin Tint & Sheer Glow Hy Charlotte Tilbury  38.40     4.4          509.0           4
Summer Fridays Sheer Skin Tint - Skin Tint with Hy    Summer Fridays  42.00     4.4          355.0           4
Palladio Skin Tint with Aloe, Niacinamide and Hyal          Palladio  15.99     4.2          174.0           4
Beauty Skin Tint for Black Women, Covebalm Foundat          Covebalm  15.88   

### 4.12 Badge Analysis (Best Seller & Amazon's Choice)


In [24]:
badge_summary = pd.DataFrame(
    {
        "Badge": ["Best Seller", "Amazon's Choice", "Amazon Prime", "Amazon Exclusive"],
        "Count": [
            df["is_best_seller"].sum(),
            df["is_amazon_choice"].sum(),
            df["is_amazon_prime"].sum(),
            df["is_amazon_exclusive"].sum(),
        ],
    }
)

fig = px.bar(
    badge_summary,
    x="Badge",
    y="Count",
    color="Badge",
    title=f"🏅 Badge Distribution Across Products (out of {len(df)})",
    template=PLOTLY_TEMPLATE,
    text="Count",
)
fig.update_layout(showlegend=False)
fig.show()

### 4.13 Bought in Past Month: Demand Signals

Amazon shows this as a rounded label such as "400+" or "2K+". We parsed the number in the label, so each value is a lower bound, and products with the same label are tied.

In [25]:
demand_df = df.dropna(subset=["bought_past_month"]).nlargest(15, "bought_past_month")[
    ["title", "brand", "bought_past_month", "price", "rating"]
]

# Shorten titles for display
demand_df["short_title"] = demand_df["title"].str[:45] + "…"

fig = px.bar(
    demand_df,
    x="bought_past_month",
    y="short_title",
    orientation="h",
    color="price",
    color_continuous_scale="RdBu_r",
    title="🔥 Top 15 Products by Units Bought in Past Month (Amazon's rounded label)",
    labels={
        "bought_past_month": "Units Bought (Past Month, lower bound)",
        "short_title": "",
        "price": "Price ($)",
    },
    template=PLOTLY_TEMPLATE,
    text="bought_past_month",
    height=600,
)
fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.show()

### 4.14 Where Does MAC Appear?

Finally, find MAC's own listings and compare them with the median product in the filtered data.

In [26]:
cols = ["title", "seller", "price", "rating", "total_reviews", "bought_past_month"]
mac = df[df["brand"].str.upper().eq("MAC")]
print(f"MAC listings: {len(mac)} of {len(df)}")
display(mac[cols].assign(title=lambda d: d["title"].str[:60]))

print("Positioning keywords mentioned in MAC's listings (1 = mentioned):")
display(mac[kw_cols].T.set_axis(mac["asin"], axis=1))

print("Median of all filtered products:")
df[["price", "rating", "total_reviews", "bought_past_month"]].median()

MAC listings: 2 of 181


,title,seller,price,rating,total_reviews,bought_past_month
18,MAC Studio Fix Powder Plus Foundation for Wome...,Prime111,39.00,4.5,438.0,200.0
28,"MAC Studio Fix Powder Plus Foundation, NW25, 0...",HNN Warehouse Deals,29.98,4.6,1191.0,200.0


Positioning keywords mentioned in MAC's listings (1 = mentioned):


asin,B0042CKBMO,B007HYQET8
full_coverage,0,1
medium_coverage,0,0
lightweight,0,0
hydrating,0,0
matte,0,1
dewy,0,0
natural_finish,0,0
spf,0,0
long_wear,0,1
skin_tint,0,0


Median of all filtered products:


price                  15.98
rating                  4.40
total_reviews        2347.00
bought_past_month     600.00
dtype: float64

---

## 🧠 Step 5: Key Insights & MAC Recommendations

These findings describe the 181 US-dollar makeup foundation listings kept after cleaning, from one snapshot of Amazon's Best Selling results.

### What the data tells us:

| Insight              | Finding |
| -------------------- | ------- |
| **Price**            | About two thirds of listings cost **under \$20** (median about \$16). Only about one in four costs \$30 or more. |
| **Brands**           | **COVERGIRL, L'Oréal Paris, and Maybelline** have the most listings. The field is fragmented: 92 brands, most with a single listing. The \$30+ listings come mostly from prestige brands such as Clinique, IT Cosmetics, Estée Lauder, and bareMinerals. |
| **MAC's presence**   | Only **two MAC listings** appear, both Studio Fix Powder Plus at \$30 to \$39, both sold by third-party sellers. Their ratings (4.5 and 4.6) are above the median, but their review counts and "bought in past month" labels are below it. |
| **Ratings**          | Ratings are compressed: the middle half of products sits between about **4.1 and 4.5**, so a high rating alone does not set a product apart. |
| **Social proof**     | Higher-rated products tend to have more reviews (moderate positive correlation). Products under \$10 have the highest median review count; above \$10 there is no clear pattern by price. |
| **Positioning**      | **Hydrating** and **lightweight** are the most common claims, mentioned by more than half of listings. **Skin tint** and **stick** formats are the least common. |
| **"Younger shopper" attributes** | Between about a fifth and a third of listings in each tier below \$50 mention three or more of the five attributes. High scores are not limited to cheap products. |
| **Badges**           | About half of the listings carry **Amazon's Choice**; only one carries **Best Seller**. |
| **Demand signals**   | The six listings labelled 10K+ or 20K+ split evenly: three established brands priced under \$20 (Maybelline, Laura Geller, COVERGIRL) and three color-changing foundations priced from about \$28 to \$44. |

### 🎯 MAC Cosmetics Action Items:

1. **Check who is selling MAC on Amazon.** Both MAC listings in these results come from third-party sellers, not from MAC. Before any listing or pricing work, confirm which listings MAC controls.
2. **Revise Amazon copy** where it is true of the product. MAC's listings do not mention *hydrating* or *lightweight*, the two most common claims among competitors.
3. **Build review volume** within Amazon's review policies. MAC's listings have well-rated but thin review histories compared with the median product.
4. **Test a lower-priced entry point** (for example, a smaller size) carefully. Most listings cost under \$20, but MAC's current price sits with the prestige brands, where ratings are also high.
5. **Consider skin-tint positioning** as a less crowded space in these results, while remembering that this data cannot show whether younger shoppers prefer it.

:::{warning} What this data cannot tell you

This is one snapshot of search results with no sales figures, no buyer demographics, and no history. "Bought in past month" is a rounded lower bound. Use these findings to choose what to test, not as proof of what drives sales.

:::
